# RNA-seq analysis: GEO GSE107011

Real public deposited bulk RNA-seq TPM data from Monaco et al. (2019), PMID 30726743, DOI 10.1016/j.celrep.2019.01.041. This is a new retrospective secondary analysis, not a reproduction of the paper’s entire deconvolution study. Original authors retain credit for generating the data.

## Setup

Run from the repository root after installing `requirements.lock`. The validation runner executes this notebook and retains a separate executed copy. All code blocks below run against the same versioned implementation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
import analysis
OUT = Path("artifacts/run")

## 1. Acquire and reconcile

Download the expression matrix, GEO sample metadata, and PubMed MEDLINE record using pinned source hashes. Match TPM column names to GEO titles by an explicit suffix rule, and require one-to-one IDs, exact sample sets, finite nonnegative values and consistent cell-type fields. Biopython parses the publication metadata; it is not a single-cell analysis library.

## 2. Quality checks and descriptive analysis

Report sample counts, donor counts, cell-type coverage, detected genes and total deposited TPM. Exclude PBMC mixtures and require at least three donors per included cell type, independent of model results. Apply log2(TPM+1), not a second raw-count normalization. PCA is exploratory and fitted on all included samples; it is never used as validation evidence.

## 3. Evaluate without donor leakage

Leave one donor out at a time. Every training fold independently fits variance filtering, ANOVA feature selection, scaling and logistic regression. Compare against a dummy baseline and a sample-level split. Report every fold and its donor overlap. Cell-type classification within four donors does not validate use in new cohorts, patients, diseases or clinical workflows.

## 4. Exact paired contrast

Compare NK cells with naive B cells within the same four donors using log2(TPM+1) differences. Enumerate all 16 sign flips. The minimum attainable two-sided exact p-value is 2/16 = 0.125. Apply Benjamini-Hochberg correction across tested genes, and distinguish descriptive effect sizes from statistical discoveries. These are exploratory transformed-expression contrasts, not raw-count differential expression with DESeq2/edgeR.

## Execute the complete analysis

Inspect the returned results and tables. No outcome is hand-written into the code or edited to create a better story.

In [ ]:
summary = analysis.run(OUT)
print(json.dumps(summary, indent=2))

In [ ]:
display(pd.read_csv(OUT / "quality_control.csv").head())
display(Image(filename=str(OUT / "pca.png")))

In [ ]:
display(pd.read_csv(OUT / "donor_holdout_scores.csv"))
display(Image(filename=str(OUT / "confusion_matrix.png")))

In [ ]:
contrast = pd.read_csv(OUT / "paired_NK_vs_naive_B.csv")
display(contrast.sort_values("mean_paired_log2_TPM_plus1_difference", ascending=False).head(10))
display(Image(filename=str(OUT / "paired_contrast.png")))

## What was validated

`tests/` checks hand-calculated examples, independent formulations, malformed inputs and relevant invariants. `python scripts/validate.py` runs pytest, the full pipeline and this notebook. Passing these checks is bounded computational evidence, not clinical validation. See `reports/reference/FINDINGS.md` for actual results and limitations.